<a href="https://colab.research.google.com/github/VenkatesanNadimuthu/C3AN-Model/blob/001-gpt2-pretrain-recipes/Experiments%5CInstruction_Fine_Tuning_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import math
import time
import os
import re
import json
import random
import matplotlib.pyplot as plt
import collections
from torch.utils.data import Dataset, DataLoader, random_split
import logging

# ==========================================
# 1. Configuration & Hardware Setup
# ==========================================

class Config:
    # --------------------------------------
    # IFT Data Configuration
    # --------------------------------------
    dataset_path = '/content/structured_recipes_finetune.jsonl'

    # --------------------------------------
    # Model Architecture
    # --------------------------------------
    block_size = 512
    vocab_size = 50304
    n_layer = 6
    n_head = 8
    n_embd = 512
    dropout = 0.1

    # --------------------------------------
    # Fine-Tuning Hyperparameters
    # --------------------------------------
    batch_size = 16
    learning_rate = 5e-5
    epochs = 5             # Changed from max_iters to epochs for Fine-Tuning
    eval_interval = 100

    # --------------------------------------
    # System
    # --------------------------------------
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
    ignore_index = -100

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

if Config.device == 'cuda':
    logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    if Config.dtype == 'bfloat16':
        logger.info("A100/Ampere detected. Using bfloat16.")

torch.manual_seed(1337)

# ==========================================
# 2. Instruction Data Pipeline
# ==========================================

class InstructionDataset(Dataset):
    """
    Consumes JSONL data for Instruction Fine-Tuning.
    Implements Masked Supervision.
    """
    def __init__(self, file_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.data = []
        self.raw_entries = []

        if not os.path.exists(file_path):
             logger.warning(f"File {file_path} not found. Generating dummy JSONL.")
             self._create_dummy_jsonl(file_path)

        logger.info(f"Loading JSONL data from {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            try:
                entry = json.loads(line)

                # 1. Format prompt
                prompt_text = self._format_prompt_only(entry)
                # 2. Format full sequence
                response_text = entry.get('response', entry.get('output', ''))
                full_text = prompt_text + response_text

                # 3. Tokenize
                prompt_tokens = tokenizer.encode(prompt_text)
                full_tokens = tokenizer.encode(full_text)
                full_tokens.append(tokenizer.eot_token)

                # 4. Create Masked Labels
                labels = full_tokens.copy()
                # Mask prompt tokens
                labels[:len(prompt_tokens)] = [Config.ignore_index] * len(prompt_tokens)

                self.data.append({
                    'input_ids': full_tokens,
                    'labels': labels
                })

                self.raw_entries.append({
                    'instruction': entry.get('instruction', ''),
                    'input': entry.get('input', ''),
                    'reference': response_text
                })
            except json.JSONDecodeError:
                continue

        logger.info(f"Loaded {len(self.data)} total samples.")

    def _create_dummy_jsonl(self, path):
        dummy_data = [
            {"instruction": "Create a healthy breakfast.", "input": "Spinach, Eggs", "response": "**Title:** Green Eggs\n**Ingredients:** Spinach, Eggs\n**Instructions:** Scramble them."},
            {"instruction": "Suggest a dessert.", "input": "Chocolate", "response": "**Title:** Choco Lava\n**Ingredients:** Chocolate, Flour\n**Instructions:** Bake it."}
        ] * 100
        with open(path, 'w') as f:
            for entry in dummy_data:
                f.write(json.dumps(entry) + '\n')

    def _format_prompt_only(self, entry):
        instruction = entry.get('instruction', '')
        input_context = entry.get('input', '')
        if input_context:
            return (f"Below is an instruction that describes a task, paired with an input that provides further context. "
                    f"Write a response that appropriately completes the request.\n\n"
                    f"### Instruction:\n{instruction}\n\n"
                    f"### Input:\n{input_context}\n\n"
                    f"### Response:\n")
        else:
            return (f"Below is an instruction that describes a task. "
                    f"Write a response that appropriately completes the request.\n\n"
                    f"### Instruction:\n{instruction}\n\n"
                    f"### Response:\n")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['input_ids']
        labels = item['labels']

        if len(tokens) > self.block_size:
            # Truncate
            x = torch.tensor(tokens[:-1][:self.block_size], dtype=torch.long)
            y = torch.tensor(labels[1:][:self.block_size], dtype=torch.long)
        else:
            # Pad
            pad_len = self.block_size - len(tokens)
            pad_token = self.tokenizer.eot_token
            x_list = tokens[:-1] + [pad_token] * (pad_len + 1)
            y_list = labels[1:] + [Config.ignore_index] * (pad_len + 1)

            x = torch.tensor(x_list[:self.block_size], dtype=torch.long)
            y = torch.tensor(y_list[:self.block_size], dtype=torch.long)

        return x, y # Moved to_device to training loop for DataLoader compatibility

# ==========================================
# 3. Model Architecture
# ==========================================

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'wpe': nn.Embedding(config.block_size, config.n_embd),
            'drop': nn.Dropout(config.dropout),
            'h': nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            'ln_f': nn.LayerNorm(config.n_embd),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=Config.ignore_index)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self.forward(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ==========================================
# 4. IEEE Evaluation Metrics & Visualizer
# ==========================================

class MetricsCalculator:
    @staticmethod
    def get_ngrams(text, n):
        words = re.findall(r'\w+', text.lower())
        return collections.Counter(zip(*[words[i:] for i in range(n)]))

    @staticmethod
    def calculate_bleu_4(candidate, reference):
        precisions = []
        for n in range(1, 5):
            cand_ngrams = MetricsCalculator.get_ngrams(candidate, n)
            ref_ngrams = MetricsCalculator.get_ngrams(reference, n)
            overlap = cand_ngrams & ref_ngrams
            count_overlap = sum(overlap.values())
            count_cand = sum(cand_ngrams.values())
            precisions.append(count_overlap / count_cand if count_cand > 0 else 0)

        if any(p == 0 for p in precisions): return 0.0
        return math.exp(sum(math.log(p) for p in precisions) / 4.0)

    @staticmethod
    def calculate_rouge_l(candidate, reference):
        c_words = re.findall(r'\w+', candidate.lower())
        r_words = re.findall(r'\w+', reference.lower())
        if not c_words or not r_words: return 0.0
        m, n = len(c_words), len(r_words)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if c_words[i - 1] == r_words[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        lcs = dp[m][n]
        if lcs == 0: return 0.0
        rec = lcs / n
        prec = lcs / m
        return 2 * (prec * rec) / (prec + rec)

class IEEE_Visualizer:
    @staticmethod
    def plot_convergence(history_data, save_path="ieee_convergence.png"):
        plt.style.use('default')
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Plot 1: Loss
        axes[0].plot(history_data['iterations'], history_data['train_loss'], label='Train Loss', alpha=0.5)
        axes[0].plot(history_data['eval_iters'], history_data['val_loss'], label='Val Loss', marker='o', linewidth=2)
        axes[0].set_title('Loss Convergence')
        axes[0].set_xlabel('Steps')
        axes[0].legend()
        axes[0].grid(True, linestyle='--', alpha=0.6)

        # Plot 2: Perplexity
        axes[1].plot(history_data['eval_iters'], history_data['ppl'], color='green', marker='s')
        axes[1].set_title('Validation Perplexity')
        axes[1].grid(True, linestyle='--', alpha=0.6)

        # Plot 3: Metrics
        metrics = ['BLEU-4', 'ROUGE-L']
        scores = [history_data['final_bleu'], history_data['final_rouge']]
        axes[2].bar(metrics, scores, color=['purple', 'red'], alpha=0.7)
        axes[2].set_title('Test Set Quality')
        axes[2].set_ylim(0, 1.0)

        plt.tight_layout()
        plt.savefig(save_path)
        print(f"IEEE Plots saved to {save_path}")

# ==========================================
# 5. Fine-Tuning Execution
# ==========================================

@torch.no_grad()
def estimate_loss(model, dataloader):
    """Estimate loss using the full validation loader."""
    model.eval()
    losses = []
    for X, Y in dataloader:
        X, Y = X.to(Config.device), Y.to(Config.device)
        with torch.amp.autocast(device_type=Config.device, dtype=getattr(torch, Config.dtype)):
            _, loss = model(X, Y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses) if losses else 0.0

def finetune():
    print("="*40)
    print(" STARTING INSTRUCTION FINE-TUNING")
    print("="*40)

    enc = tiktoken.get_encoding("gpt2")
    full_ds = InstructionDataset(Config.dataset_path, enc, Config.block_size)

    # Random Split for unbiased validation
    train_size = int(0.9 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])

    # Use DataLoaders for proper epoch-based training
    train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.batch_size, shuffle=False)

    model = SLM(Config).to(Config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.epochs * len(train_loader))
    scaler = torch.amp.GradScaler(device=Config.device, enabled=(Config.dtype != 'float32'))

    history = {'iterations': [], 'train_loss': [], 'eval_iters': [], 'val_loss': [], 'ppl': []}

    best_val_loss = float('inf')
    global_step = 0
    start_time = time.time()

    for epoch in range(Config.epochs):
        print(f"\nEpoch {epoch+1}/{Config.epochs}")

        for X, Y in train_loader:
            X, Y = X.to(Config.device), Y.to(Config.device)

            # Training Step
            with torch.amp.autocast(device_type=Config.device, dtype=getattr(torch, Config.dtype)):
                logits, loss = model(X, Y)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            history['iterations'].append(global_step)
            history['train_loss'].append(loss.item())

            # Validation
            if global_step % Config.eval_interval == 0:
                val_loss = estimate_loss(model, val_loader)
                ppl = math.exp(val_loss)
                history['eval_iters'].append(global_step)
                history['val_loss'].append(val_loss)
                history['ppl'].append(ppl)

                print(f"Step {global_step}: Train Loss {loss.item():.4f}, Val Loss {val_loss:.4f}, PPL {ppl:.2f}")

                # Save Best Model
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), 'best_recipe_slm.pth')
                    print("  -> New best model saved.")

            global_step += 1

    print(f"Fine-Tuning Finished in {time.time()-start_time:.2f}s")

    # Load best model for final eval
    model.load_state_dict(torch.load('best_recipe_slm.pth'))

    print("Running Final IEEE Quality Evaluation on Validation Set...")
    # Helper to get raw entries from subset for metric calc
    val_indices = val_ds.indices
    val_raw_entries = [full_ds.raw_entries[i] for i in val_indices]

    avg_bleu, avg_rouge = evaluate_generation_quality(model, val_raw_entries, enc)
    history['final_bleu'] = avg_bleu
    history['final_rouge'] = avg_rouge
    print(f"Final Results -> BLEU-4: {avg_bleu:.4f} | ROUGE-L: {avg_rouge:.4f}")

    IEEE_Visualizer.plot_convergence(history)
    return model, enc

@torch.no_grad()
def evaluate_generation_quality(model, raw_entries, tokenizer, num_samples=30):
    model.eval()
    bleu_scores = []
    rouge_scores = []

    samples = random.sample(raw_entries, min(num_samples, len(raw_entries)))

    for entry in samples:
        if entry['input']:
            prompt = f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{entry['instruction']}\n\n### Input:\n{entry['input']}\n\n### Response:\n"
        else:
            prompt = f"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\n{entry['instruction']}\n\n### Response:\n"

        idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=Config.device)[None, ...]

        try:
            y = model.generate(idx, max_new_tokens=150, temperature=0.7)
            output_text = tokenizer.decode(y[0].tolist())
            generated_response = output_text[len(prompt):]

            b4 = MetricsCalculator.calculate_bleu_4(generated_response, entry['reference'])
            rl = MetricsCalculator.calculate_rouge_l(generated_response, entry['reference'])
            bleu_scores.append(b4)
            rouge_scores.append(rl)
        except Exception:
            continue

    model.train()
    return (sum(bleu_scores)/len(bleu_scores) if bleu_scores else 0,
            sum(rouge_scores)/len(rouge_scores) if rouge_scores else 0)

# ==========================================
# 6. Interactive Test
# ==========================================

def test_model(model, enc):
    print("\n" + "="*40)
    print(" INSTRUCTION MODE TEST")
    print("="*40)

    test_instruction = "Create a spicy pasta dish."
    test_input = "Tomatoes, Chilies, Garlic"

    prompt = f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{test_instruction}\n\n### Input:\n{test_input}\n\n### Response:\n"
    idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=Config.device)[None, ...]

    print(f"Instruction: {test_instruction}")
    print(f"Context: {test_input}")
    print("\nGenerating...", end="", flush=True)

    model.eval()
    with torch.no_grad():
        y = model.generate(idx, max_new_tokens=200, temperature=0.7)

    output = enc.decode(y[0].tolist())
    print("\n" + "-"*40)
    print(output[len(prompt):])
    print("-"*40)

if __name__ == "__main__":
    import matplotlib
    matplotlib.use('Agg')
    ft_model, tokenizer = finetune()
    test_model(ft_model, tokenizer)

 STARTING INSTRUCTION FINE-TUNING

Epoch 1/5
Step 0: Train Loss 10.9275, Val Loss 10.4454, PPL 34387.25
  -> New best model saved.
Step 100: Train Loss 5.9020, Val Loss 5.9087, PPL 368.24
  -> New best model saved.
Step 200: Train Loss 4.5628, Val Loss 4.5207, PPL 91.90
  -> New best model saved.
Step 300: Train Loss 4.3220, Val Loss 3.9467, PPL 51.77
  -> New best model saved.
Step 400: Train Loss 3.7216, Val Loss 3.5946, PPL 36.40
  -> New best model saved.
Step 500: Train Loss 3.3656, Val Loss 3.3380, PPL 28.16
  -> New best model saved.
Step 600: Train Loss 3.0508, Val Loss 3.1625, PPL 23.63
  -> New best model saved.
Step 700: Train Loss 3.4596, Val Loss 3.0311, PPL 20.72
  -> New best model saved.
Step 800: Train Loss 2.7338, Val Loss 2.9173, PPL 18.49
  -> New best model saved.
Step 900: Train Loss 2.8500, Val Loss 2.8291, PPL 16.93
  -> New best model saved.
Step 1000: Train Loss 2.8826, Val Loss 2.7360, PPL 15.42
  -> New best model saved.
Step 1100: Train Loss 2.6767, Val Los